# Just enough linear algebra

MichAl Academy, lesson 1.7.

Run each cell with **Shift+Enter**. numpy only.

One operation, one ratio, and then a tiny search engine built out of both.

## 1. The dot product

Multiply the matching entries, add up the results.

In [ ]:
import numpy as np

a = np.array([3, 2])
b = np.array([6, 4])

by_hand = a[0] * b[0] + a[1] * b[1]

print("by hand :", by_hand)
print("a @ b   :", a @ b)
print("np.dot  :", np.dot(a, b))

Three ways of writing the same arithmetic. `@` is the one you will see most.

That single number mixes two different facts: how much the vectors agree in
direction, and how long they are. Watch what happens when only the length
changes.

In [ ]:
short = np.array([3, 2])
long  = np.array([6, 4])      # same direction, twice as far

print("a @ short :", a @ short)
print("a @ long  :", a @ long, " <- doubled, and nothing about the direction changed")

## 2. Length, and dividing it back out

`np.linalg.norm` is the length of a vector: the straight-line distance from the
origin to its tip.

In [ ]:
print("|a|     :", np.linalg.norm(a))
print("|short| :", np.linalg.norm(short))
print("|long|  :", np.linalg.norm(long))
print()
print("check by Pythagoras:", (3**2 + 2**2) ** 0.5)

In [ ]:
def cosine(u, v):
    return (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))

for name, v in [("same direction, longer", long),
                ("identical",              short),
                ("at right angles",        np.array([-2, 3])),
                ("opposite",               np.array([-3, -2]))]:
    c = cosine(a, v)
    print(f"{name:24s} dot {a @ v:4d}   cos {c:+.3f}   angle {np.degrees(np.arccos(np.clip(c, -1, 1))):5.1f} deg")

Look at the first two lines. Very different dot products, **identical cosine**,
because one vector is simply twice as long as the other and points the same way.

Cosine runs from 1 (same direction) through 0 (at right angles, carrying no
information about each other) to -1 (opposite).

## 3. Nothing changes in higher dimensions

You cannot picture twelve dimensions. You do not need to: the arithmetic is
identical and so is the meaning.

In [ ]:
rng = np.random.default_rng(5)

p = rng.normal(size=12)
q = rng.normal(size=12)

print("dot   :", round(p @ q, 3))
print("cos   :", round(cosine(p, q), 3))
print()
print("two random directions in 12 dimensions are close to at right angles,")
print("which stops being surprising once you notice how much room there is")

## 4. A search engine, in about ten lines

Turn each document into a vector by counting words. Rank by cosine against the
query. That is the whole idea behind retrieval, and Track 5 replaces the word
counts with embeddings without changing anything else.

In [ ]:
DOCS = [
    "phishing url detection model",
    "url shortener abuse report",
    "malware sandbox detonation",
    "phishing awareness training and phishing simulation and phishing reporting "
    "and url handling and url filtering and detection of phishing and detection "
    "of malware and staff training and reporting",
]

QUERY = "phishing url detection"


def vectorise(texts):
    vocab = sorted({w for t in texts for w in t.split()})
    index = {w: i for i, w in enumerate(vocab)}
    out = np.zeros((len(texts), len(vocab)))
    for r, t in enumerate(texts):
        for w in t.split():
            out[r, index[w]] += 1
    return out, vocab


matrix, vocab = vectorise(DOCS + [QUERY])
doc_vecs, query_vec = matrix[:-1], matrix[-1]

print("vocabulary size:", len(vocab))
print("document vectors shape:", doc_vecs.shape)

In [ ]:
print(f"{'doc':>4}  {'words':>5}  {'dot':>5}  {'cos':>6}   text")
for i, d in enumerate(DOCS):
    dot = doc_vecs[i] @ query_vec
    cos = cosine(doc_vecs[i], query_vec)
    print(f"{i:>4}  {int(doc_vecs[i].sum()):>5}  {dot:>5.0f}  {cos:>6.3f}   {d[:52]}")

Two different rankings from the same numbers.

By raw dot product the long document wins, because it says "phishing" and "url"
many times and length alone drives the score up. By cosine the short, focused
document wins, because cosine asks what the document is *about* rather than how
much of it there is.

That is the whole reason retrieval uses cosine.

## 5. Matrices, briefly

A matrix is a stack of vectors. Multiplying it by one vector takes the dot
product of that vector with every row at once.

In [ ]:
scores = doc_vecs @ query_vec        # one dot product per document, in one go

print("doc_vecs   ", doc_vecs.shape)
print("query_vec  ", query_vec.shape)
print("scores     ", scores.shape, "->", scores)

`(4, n) @ (n,)` gives `(4,)`. That is the broadcasting and shape rule from lesson
1.3 arriving from the other side, and it is exactly what one layer of a neural
network does: a matrix multiply, then one non-linear function.

## 6. Your turn

Below is a ranking function with a bug. It is supposed to return the documents
most *about* the query, and it returns the longest one.

Change one line.

In [ ]:
def rank(docs_matrix, query, top=2):
    scores = docs_matrix @ query      # TODO: this is not similarity
    order = np.argsort(scores)[::-1]
    return [(int(i), round(float(scores[i]), 3)) for i in order[:top]]


best = rank(doc_vecs, query_vec)

print("ranking:", best)
print()
for i, s in best:
    print(f"  doc {i} (score {s}): {DOCS[i][:60]}")
print()
print("is the short focused document first?", best[0][0] == 0)

The scores are dot products. Ask what is missing before they become similarities.

<details>
<summary>Answer</summary>

Divide by both lengths. The query length is the same for every document so it
does not change the order, but the document lengths very much do.

```python
def rank(docs_matrix, query, top=2):
    lengths = np.linalg.norm(docs_matrix, axis=1) * np.linalg.norm(query)
    scores = (docs_matrix @ query) / lengths
    order = np.argsort(scores)[::-1]
    return [(int(i), round(float(scores[i]), 3)) for i in order[:top]]
```

`np.linalg.norm(docs_matrix, axis=1)` gives one length per row, shape `(4,)`,
which then broadcasts against the `(4,)` of scores. Same rule as lesson 1.3.

</details>

## What you now have

- A vector is a list of numbers and also a direction, and both readings are useful
- The dot product is multiply-and-add, written `@`
- It mixes direction with length, so on its own it is hard to read
- Divide by both lengths and you get cosine, which is direction only, from 1 to -1
- Cosine is what "similar" means for search, for embeddings and for retrieval
- A matrix multiply is a dot product against every row at once, which is one network layer

Next is lesson 1.8, probability, and the conditional probability that decides
whether a detector is worth switching on.